# Érzelemfelismerő rendszer nulláról demóig

**MI alapú ember-gép interakció · BME VIK · 90 perces gyakorlat**

Ma egy teljes beszédalapú érzelemfelismerő rendszert építünk fel — adattól a
működő demóig. Nem az a cél, hogy versenyképes modellt kapjunk, hanem hogy a
**teljes láncot** egyszer végigcsináld, és lásd, hol dől el a teljesítmény.

| # | blokk | idő |
|---|---|---|
| 0 | Indulás | 5' |
| 1 | Az adat: mit is akarunk felismerni | 12' |
| 2 | Kézi jellemzők + első modell | 15' |
| 3 | A csapda: beszélőfüggő vs. beszélőfüggetlen | 12' |
| 4 | Önfelügyelt embedding (WavLM) | 20' |
| 5 | Kiértékelés rendesen | 10' |
| 6 | Demó: a saját hangod | 12' |
| — | Zárás | 4' |

**Előfeltétel:** a `00_elokeszites.ipynb` lefuttatva.
**Futtatókörnyezet:** T4 GPU.

---
## 0 · Indulás

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

WORK = "/content/drive/MyDrive/ser_gyakorlat"
os.environ["HF_HOME"] = os.path.join(WORK, "hf")
print("munkakönyvtár:", WORK)

In [ ]:
# --- a segédmodul betöltése ---------------------------------------------
# Ha a letöltés nem megy, töltsd fel a ser_utils.py-t kézzel a bal oldali
# fájlpanelen (mappa ikon → feltöltés), és futtasd újra ezt a cellát.
import os, sys, subprocess, importlib

SER_UTILS_URL = "https://<a-tárgy-honlapja>/ser_utils.py"

def betolt_ser_utils(url=SER_UTILS_URL, fajl="ser_utils.py"):
    megvan = os.path.exists(fajl) and os.path.getsize(fajl) > 1000
    if not megvan:
        if "<" in url:
            raise RuntimeError(
                "A SER_UTILS_URL még helyőrző cím.\n"
                "Írd át a tényleges letöltési helyre, VAGY töltsd fel a "
                "ser_utils.py-t kézzel a bal oldali fájlpanelen, majd futtasd "
                "újra ezt a cellát.")
        ideiglenes = fajl + ".letoltes"
        kod = subprocess.run(["wget", "-q", "-O", ideiglenes, url]).returncode
        meret = os.path.getsize(ideiglenes) if os.path.exists(ideiglenes) else 0
        if kod != 0 or meret < 1000:
            if os.path.exists(ideiglenes):
                os.remove(ideiglenes)
            raise RuntimeError(
                f"A letöltés nem sikerult: {url} (wget kód: {kod}, {meret} bájt).\n"
                "Töltsd fel a ser_utils.py-t kézzel a bal oldali fájlpanelen, "
                "majd futtasd újra ezt a cellát.")
        os.replace(ideiglenes, fajl)

    szoveg = open(fajl, encoding="utf-8").read()
    if "def gpu_info" not in szoveg:
        raise RuntimeError(
            f"A {fajl} létezik ({len(szoveg)} bájt), de nem a segédmodul van "
            "benne. Töröld a fájlt, és töltsd fel újra.")

    sys.modules.pop("ser_utils", None)   # a korábbi (esetleg üres) modul eldobása
    return importlib.import_module("ser_utils")

su = betolt_ser_utils()
su.self_check()

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

In [ ]:
wav_root = su.download_ravdess(WORK)     # ha megvan, csak beolvas
df = su.scan_ravdess(wav_root)
print(df.shape)
df.head()

---
## 1 · Az adat: mit is akarunk felismerni

**RAVDESS** — 24 színész, 1440 felvétel, 8 érzelemkategória. Két rögzített
angol mondat, tehát a nyelvi tartalom minden felvételnél ugyanaz: amit a
modell megtanulhat, az csak a *hogyan*, nem a *mit*.

A metaadat a fájlnévben van — nem kell külön annotációs fájl:

```
03-01-05-02-01-01-12.wav
 │  │  │  │  │  │  └─ színész (12)
 │  │  │  │  │  └──── ismétlés
 │  │  │  │  └─────── mondat
 │  │  │  └────────── intenzitás (erős)
 │  │  └───────────── érzelem (harag)
 │  └──────────────── beszéd
 └─────────────────── csak audio
```

In [ ]:
# osztályeloszlás és beszélőnkénti eloszlás
fig, axes = plt.subplots(1, 2, figsize=(13, 3.6))

counts = df.emotion.value_counts().reindex(su.EMOTION_ORDER).dropna()
axes[0].bar(counts.index, counts.values, color="#1B4965")
axes[0].set_title("Felvételek érzelemkategóriánként")
axes[0].tick_params(axis="x", rotation=40)
axes[0].set_ylabel("darab")

per_actor = df.groupby("actor").size()
axes[1].bar(per_actor.index, per_actor.values, color="#9DB9C9")
axes[1].set_title("Felvételek színészenként")
axes[1].set_xlabel("színész azonosító")
plt.tight_layout(); plt.show()

print(df.emotion.value_counts().to_string())

> **Kérdés a teremhez:** a „semleges” kategóriából feleannyi felvétel van, mint
> a többiből. Miért, és mit fog ez okozni a tanításnál?
>
> *(A RAVDESS-ben a semlegeshez nincs „erős” intenzitás — ezért fele annyi.
> Következmény: kiegyensúlyozatlan adat, tehát az accuracy félrevezető lesz,
> és `class_weight="balanced"` kell.)*

In [ ]:
# Ugyanaz a színész, ugyanaz a mondat, két különböző érzelem.
sel = df[(df.actor == 3) & (df.statement == 1) & (df.repetition == 1)]
for emo in ["semleges", "harag"]:
    row = sel[sel.emotion == emo].iloc[0]
    y = su.load_wav(row.path)
    print(f"--- {emo} (színész {row.actor}, {row.intensity} intenzitás)")
    su.play(y)

In [ ]:
# és ugyanez ránézésre
for emo in ["semleges", "harag"]:
    row = sel[sel.emotion == emo].iloc[0]
    y = su.load_wav(row.path)
    su.plot_waveform_spectrogram(y, title=f"{emo} — színész {row.actor}")
    plt.show()

> **Kérdés:** mi különbözik a két felvételen? Mit lát ebből egy MFCC-alapú
> jellemzőkészlet, és mit nem?
>
> *(Hangerő, beszédtempó, alapfrekvencia-tartomány és a spektrum meredeksége
> jól látszik. Amit nem lát: a kontextus és a szándék.)*

---
## 2 · Kézi jellemzők és az első modell

Most az előadáson látott utat járjuk be: **alacsonyszintű jellemzők →
statisztikai függvények → beszédszintű vektor → osztályozó**.

Az `eGeMAPSv02` a terület szabványos kis jellemzőkészlete: 88 érték
felvételenként (alapfrekvencia, jitter, shimmer, formánsok, energia,
spektrális jellemzők statisztikái).

In [ ]:
%%time
# az összes felvétel beolvasása memóriába — kb. 1 perc
waves = su.load_all(df.path.tolist())
print("egy felvétel hossza átlagosan: %.2f s" % np.mean([len(w)/su.SR for w in waves]))

In [ ]:
%%time
X_hand, feat_names = su.egemaps_features(waves)
y = df.emotion.values
print("jellemzőmátrix:", X_hand.shape)
print("első néhány jellemző:", feat_names[:6])

### Első kísérlet — véletlen felosztással

Osszuk fel az adatot véletlenszerűen 80/20 arányban, és nézzük meg, mit tud
egy SVM. **Ez a felosztás hibás lesz** — de előbb lássuk a számot.

In [ ]:
tr_rand, te_rand = su.random_split(df, test_size=0.2, seed=0)
r_rand = su.run_experiment(X_hand, y, tr_rand, te_rand,
                           name="kézi jellemzők + véletlen split")

> **Kérdés a teremhez:** elégedettek lennénk ezzel az eredménnyel? Mit
> jelentene, ha ezt a számot beírnánk egy szakdolgozatba?

---
## 3 · A csapda: beszélőfüggő vs. beszélőfüggetlen felosztás

A RAVDESS-ben **színészenként 60 felvétel** van. Véletlen felosztásnál
ugyanannak a színésznek egy része tanításba, másik része tesztbe kerül —
így a modell megtanulhatja **a beszélőt** a feladat helyett.

Csináljuk meg rendesen: a teszt színészei (21–24) **sosem** szerepelnek
tanításban. Ugyanaz a jellemzőkészlet, ugyanaz a modell — csak a felosztás más.

In [ ]:
tr_spk, te_spk = su.speaker_split(df, test_actors=(21, 22, 23, 24))
print("tanító színészek:", sorted(df.actor[tr_spk].unique()))
print("teszt színészek: ", sorted(df.actor[te_spk].unique()))

r_spk = su.run_experiment(X_hand, y, tr_spk, te_spk,
                          name="kézi jellemzők + beszélőfüggetlen")

In [ ]:
su.compare_results([r_rand, r_spk],
                   title="Ugyanaz a modell, ugyanaz a jellemzőkészlet — más felosztás")
plt.show()

> **Ez a gyakorlat legfontosabb fóliája.** A különbség nem a modellből jön,
> hanem abból, hogy mit engedtünk meg magunknak a kiértékelésben.
>
> Ha egy cikkben vagy szakdolgozatban 8 osztályon 90% feletti eredményt látsz
> spontán beszéden, az első kérdés mindig ez legyen: **hogyan osztották fel az
> adatot?**

In [ ]:
su.plot_confusion(r_spk, title="Kézi jellemzők, beszélőfüggetlen felosztás")
plt.show()

---
## 4 · Önfelügyelt embedding: WavLM

Most lecseréljük a reprezentációt. A **WavLM** több tízezer óra címkézetlen
beszéden tanult önfelügyelt modell; mi **nem hangoljuk**, csak jellemzőkinyerőként
használjuk: minden felvételre kiszámoljuk a rejtett állapotok időbeli átlagát.

Az osztályozó és a felosztás **ugyanaz marad** — így tisztán a reprezentáció
hatását mérjük.

In [ ]:
%%time
# minden rejtett réteget eltárolunk, hogy utána össze tudjuk hasonlítani őket
# (13 réteg × 1440 felvétel × 768 dimenzió ≈ 57 MB)
E = su.wavlm_embeddings(waves, layers="all", batch_size=16)
print("embedding tömb:", E.shape)

In [ ]:
LAYER = 6          # középső réteg — az előadás szerint itt szokott lenni az érzelem
X_ssl = E[LAYER]

r_ssl = su.run_experiment(X_ssl, y, tr_spk, te_spk,
                          name=f"WavLM {LAYER}. réteg + beszélőfüggetlen")

In [ ]:
su.compare_results([r_rand, r_spk, r_ssl], title="Három kísérlet egymás mellett")
plt.show()

### Melyik réteg kódolja az érzelmet?

Az előadáson szó volt róla, hogy egy SSL-modell rétegei különböző információt
hordoznak: az alsók inkább akusztikai és beszélőjellemzőket, a felsők inkább
fonetikai és nyelvi tartalmat. Nézzük meg **mérésre**.

In [ ]:
%%time
layer_scores = {}
for L in range(E.shape[0]):
    r = su.run_experiment(E[L], y, tr_spk, te_spk, name=f"réteg {L}", verbose=False)
    layer_scores[L] = r["macro_f1"]
    print(f"  réteg {L:2d}: makro-F1 = {r['macro_f1']:.3f}")

In [ ]:
su.plot_layer_scan(layer_scores)
plt.show()

best_layer = max(layer_scores, key=layer_scores.get)
print(f"A legjobb réteg: {best_layer} (makro-F1 = {layer_scores[best_layer]:.3f})")

> **Kérdés:** miért nem a legfelső réteg a legjobb?
>
> *(A felső rétegek a nyelvi tartalom felé specializálódnak — a maszkolt
> predikciós tanítási cél miatt. Az érzelmet hordozó prozódiai információ a
> középső rétegekben van a legerősebben. Ezért használnak az éles rendszerek
> tanulható súlyú rétegkombinációt egyetlen réteg helyett.)*

In [ ]:
# a legjobb réteggel újra, és ezt visszük tovább a demóhoz
X_best = E[best_layer]
r_best = su.run_experiment(X_best, y, tr_spk, te_spk,
                           name=f"WavLM {best_layer}. réteg (legjobb)")

---
## 5 · Kiértékelés rendesen

Egyetlen szám sosem elég. Nézzük meg, **melyik osztály** megy rosszul, és
**mivel keveredik**.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
su.plot_confusion(r_spk, ax=axes[0], title="Kézi jellemzők")
su.plot_confusion(r_best, ax=axes[1], title=f"WavLM {best_layer}. réteg")
plt.tight_layout(); plt.show()

In [ ]:
from sklearn.metrics import f1_score
import pandas as pd

labels = [l for l in su.EMOTION_ORDER if l in set(r_best["y_true"])]
tab = pd.DataFrame({
    "kézi jellemzők": f1_score(r_spk["y_true"], r_spk["y_pred"],
                               labels=labels, average=None, zero_division=0),
    f"WavLM {best_layer}": f1_score(r_best["y_true"], r_best["y_pred"],
                                    labels=labels, average=None, zero_division=0),
}, index=labels).round(3)
tab["változás"] = (tab.iloc[:, 1] - tab.iloc[:, 0]).round(3)
tab.sort_values("változás")

> **Kérdés:** melyik osztályon segített legtöbbet az SSL-reprezentáció, és
> melyiken alig? Mit mond ez arról, hogy mit tanult meg a WavLM?
>
> **Fontos tartani észben:** ez színészi, felolvasott, angol nyelvű adat, két
> rögzített mondattal. Az előadáson látott Interspeech 2025 challenge spontán
> beszéden 8 osztályon **0,43 makro-F1**-nél tartott. Amit itt mérünk, az egy
> jóval könnyebb feladat.

---
## 6 · Demó: a saját hangod

Tanítsuk újra a legjobb beállítást az **összes** adaton, aztán próbáljuk ki
élőben. A cella engedélyt fog kérni a mikrofonhoz.

In [ ]:
# teljes adaton tanított modell a demóhoz
final_model = su.make_classifier("svm")
final_model.fit(X_best, y)
print("Kész. Osztályok:", list(final_model.classes_))

embed_fn = lambda w: su.wavlm_embeddings(w, layers=best_layer, show_progress=False)

In [ ]:
# 1. próba — mondd angolul, színészesen: "Kids are talking by the door"
path = su.record_audio(seconds=4)
y_rec = su.load_wav(path)
su.play(y_rec)
su.predict_one(y_rec, final_model, embed_fn)

In [ ]:
# 2. próba — most magyarul, ahogy normálisan beszélsz
path = su.record_audio(seconds=4)
y_rec = su.load_wav(path)
su.play(y_rec)
su.predict_one(y_rec, final_model, embed_fn)

> **Ez a gyakorlat második legfontosabb tanulsága.** A modell angol, színészi,
> stúdióban felvett, eltúlzott érzelmű beszéden tanult. A te magyar, spontán,
> laptopmikrofonos felvételed **minden dimenzióban** eltér ettől:
> nyelv, beszédstílus, csatorna, érzelmi intenzitás.
>
> Ezért lesz gyenge — és ezért nem szabad egy korpuszon mért eredményt
> általános képességnek tekinteni.

---
## 7 · Más reprezentációk: mennyit javít egy nagyobb vagy célzottabb modell?

Eddig egyetlen modellel dolgoztunk. Most **ugyanazon a felosztáson, ugyanazzal
az osztályozóval** kipróbálunk többet, és mindegyiknél megkeressük a legjobb
réteget. Így tisztán a reprezentáció hatását mérjük.

| modell | paraméter | mire tanították |
|---|---|---|
| `facebook/wav2vec2-base` | 95 M | kontrasztív, maszkolt predikció |
| `microsoft/wavlm-base-plus` | 95 M | + denoising és beszélőkeverés |
| `microsoft/wavlm-large` | 317 M | ugyanaz, nagyobb és több adaton |
| `emotion2vec` *(opcionális)* | — | **érzelem-specifikus** előtanítás |

> **Futásidő.** A `wavlm-large` 24 rétegű, 1024 dimenziós: T4 GPU-n ~5 perc,
> 4 GB-os laptop GPU-n ~15-20 perc, CPU-n nagyon sokáig. Ha szorít az idő,
> vedd ki a listából, vagy szűkítsd az adatot:
> `df = df[df.actor <= 12]` és számold újra a `waves`-t.

In [ ]:
MODELLEK = [
    "facebook/wav2vec2-base",
    "microsoft/wavlm-base-plus",
    "microsoft/wavlm-large",     # ← ezt vedd ki, ha kevés az idő vagy a VRAM
]

eredmenyek = {}
for nev in MODELLEK:
    print(f"\n=== {nev}")
    Em = su.wavlm_embeddings(waves, model_name=nev, layers="all")
    L, pontok = su.best_layer_score(Em, y, tr_spk, te_spk, show_progress=False)
    eredmenyek[nev] = {
        "legjobb_reteg": L,
        "retegek": Em.shape[0],
        "dim": Em.shape[2],
        "X": Em[L].copy(),          # csak a legjobb réteget tartjuk meg
        "pontok": pontok,
    }
    print(f"  rétegek: {Em.shape[0]} · dimenzió: {Em.shape[2]} · "
          f"legjobb réteg: {L} (makro-F1 = {pontok[L]:.3f})")
    del Em                          # a teljes tömböt eldobjuk, hogy legyen memória

In [ ]:
# minden modell a saját legjobb rétegével, ugyanazon a felosztáson
futasok = [r_spk]                   # a kézi jellemzős alapvonal összehasonlításnak
for nev, d in eredmenyek.items():
    rovid = nev.split("/")[-1]
    futasok.append(su.run_experiment(d["X"], y, tr_spk, te_spk,
                                     name=rovid, verbose=False))

su.compare_results(futasok, title="Reprezentációk összehasonlítása (beszélőfüggetlen)")
plt.show()

for r in futasok:
    print(f"{r['name']:<26s} acc={r['accuracy']:.3f}  makro-F1={r['macro_f1']:.3f}")

In [ ]:
# a rétegprofilok egymáson — a nagyobb modellnél máshol van az optimum
fig, ax = plt.subplots(figsize=(9, 4))
for nev, d in eredmenyek.items():
    retegek = sorted(d["pontok"])
    # relatív rétegmélység, hogy a 12 és a 24 rétegű modell összevethető legyen
    x = [L / (len(retegek) - 1) for L in retegek]
    ax.plot(x, [d["pontok"][L] for L in retegek], "o-", ms=4,
            label=f"{nev.split('/')[-1]} ({d['retegek']-1} réteg)")
ax.set_xlabel("relatív rétegmélység (0 = konvolúciós enkóder, 1 = legfelső réteg)")
ax.set_ylabel("makro-F1")
ax.set_title("Hol van az érzelmi információ a különböző modellekben?")
ax.legend(frameon=False, fontsize=9.5)
ax.grid(alpha=.2); ax.set_axisbelow(True)
plt.show()

### Opcionális: emotion2vec

Az `emotion2vec` nem általános beszédmodell, hanem **érzelem-specifikus**
önfelügyelt előtanítással készült. Nem transformers-modell, a FunASR csomag
kell hozzá:

```
pip install -U funasr modelscope
```

Ha nem akarod telepíteni, ugord át ezt a cellát — a fenti összehasonlítás
önmagában is megválaszolja a kérdést.

In [ ]:
try:
    X_e2v = su.emotion2vec_embeddings(waves)
    r_e2v = su.run_experiment(X_e2v, y, tr_spk, te_spk, name="emotion2vec",
                              verbose=False)
    su.compare_results(futasok + [r_e2v],
                       title="…és az érzelem-specifikus előtanítás")
    plt.show()
    print(f"emotion2vec: acc={r_e2v['accuracy']:.3f}  "
          f"makro-F1={r_e2v['macro_f1']:.3f}")
except RuntimeError as e:
    print("Kihagyva:", e)

> **Kérdések:**
>
> 1. Mennyit hozott a `base` → `large` váltás? Arányos-e a paraméterszám
>    háromszorozódásával?
> 2. A `wav2vec2-base` és a `wavlm-base-plus` ugyanakkora. Miért jobb mégis az
>    egyik? *(A WavLM tanításában denoising és beszélőkeverés is volt — ettől
>    a paralingvisztikai információ jobban megmarad.)*
> 3. A rétegprofil-ábrán hol van az optimum a nagy modellnél a kicsihez képest?
> 4. Ha az `emotion2vec` kevés adattal is jól megy, mit mond ez arról, hogy
>    mikor éri meg területspecifikus előtanítást csinálni?

---
## 8 · Leave-one-speaker-out: mekkora a szórás a beszélők között?

Eddig **négy** rögzített színészen teszteltünk. Ez egy szám — de mennyire
megbízható? Lehet, hogy pont könnyű vagy pont nehéz négyest fogtunk ki.

A szigorúbb változat: **minden beszélő egyszer teszthalmaz**, a többi tanító.
24 színész → 24 forduló. Nemcsak átlagot kapunk, hanem azt is látjuk, mennyire
ingadozik a teljesítmény beszélőnként.

In [ ]:
%%time
# Melyik reprezentációval dolgozzunk? Ha lefuttattad a 7. blokkot, az ott
# nyertes modellt vesszük; különben a 4. blokk legjobb WavLM-rétegét.
if "eredmenyek" in globals() and eredmenyek:
    nyertes = max(eredmenyek,
                  key=lambda k: eredmenyek[k]["pontok"][eredmenyek[k]["legjobb_reteg"]])
    X_sel = eredmenyek[nyertes]["X"]
    print(f"reprezentáció: {nyertes}, {eredmenyek[nyertes]['legjobb_reteg']}. réteg")
else:
    X_sel = X_best
    print(f"reprezentáció: WavLM-base-plus, {best_layer}. réteg (4. blokk)")
print("alak:", X_sel.shape)

pontszamok = su.leave_one_speaker_out(X_sel, y, df.actor.values,
                                      show_progress=False)
pontszamok.describe()[["accuracy", "macro_f1"]].round(3)

In [ ]:
atlag = pontszamok.macro_f1.mean()
szoras = pontszamok.macro_f1.std(ddof=1)
print(f"LOSO makro-F1: {atlag:.3f} ± {szoras:.3f}")
print(f"legrosszabb beszélő: {pontszamok.macro_f1.min():.3f}")
print(f"legjobb beszélő:     {pontszamok.macro_f1.max():.3f}")
print(f"terjedelem:          {pontszamok.macro_f1.max() - pontszamok.macro_f1.min():.3f}")
print(f"\nÖsszehasonlításul a rögzített 4 színészes teszt: {r_best['macro_f1']:.3f}")

su.plot_speaker_scores(pontszamok)
plt.show()

In [ ]:
# nem szerinti bontás — a RAVDESS-ben a páratlan azonosító férfi, a páros nő
nem = np.where(pontszamok.beszelo % 2 == 1, "férfi", "nő")
su.plot_speaker_scores(pontszamok, csoport=nem,
                       title="LOSO beszélőnként, nem szerinti bontásban")
plt.show()

import pandas as pd
osszegzes = (pontszamok.assign(nem=nem)
             .groupby("nem")[["accuracy", "macro_f1"]]
             .agg(["mean", "std", "count"]).round(3))
osszegzes

> **Kérdések:**
>
> 1. Mekkora a szórás a beszélők között? Nagyobb-e, mint a 4. blokkban mért
>    különbség a kézi jellemzők és a WavLM között? *(Ha igen, az azt jelenti,
>    hogy egyetlen felosztásból mért különbség önmagában nem bizonyít semmit.)*
> 2. Esik-e a rögzített négy színészes eredmény a LOSO átlag ± szórás sávjába?
> 3. Van-e rendszeres különbség a férfi és a női beszélők között? Ha igen, az
>    adatból jön vagy a modellből? Hogyan lehetne ezt eldönteni?
> 4. Mit írnál egy cikkbe: az átlagot, az átlag ± szórást, vagy a
>    konfidenciaintervallumot?
>
> **Ez a blokk a 3. blokk folytatása.** Ott azt láttuk, hogy a *felosztás módja*
> számít. Itt azt, hogy a *felosztás konkrét megválasztása* is számít.

---
## 9 · Osztályok összevonása: mennyivel lesz könnyebb a feladat?

A tévesztési mátrixon látszott, hogy a **semleges** és a **nyugodt** osztály
keveredik egymással. Ez nem meglepő: a RAVDESS „calm" kategóriája lényegében
visszafogott semleges beszéd.

Mi történik, ha összevonjuk őket? A feladat 8 osztályról 7-re csökken.

In [ ]:
y7 = su.merge_labels(y, {"semleges+nyugodt": ["semleges", "nyugodt"]})
print("osztályok:", len(set(y)), "→", len(set(y7)))
import pandas as pd
print(pd.Series(y7).value_counts().to_string())

In [ ]:
# Melyik reprezentációval dolgozzunk? Ha lefuttattad a 7. blokkot, az ott
# nyertes modellt vesszük; különben a 4. blokk legjobb WavLM-rétegét.
if "eredmenyek" in globals() and eredmenyek:
    nyertes = max(eredmenyek,
                  key=lambda k: eredmenyek[k]["pontok"][eredmenyek[k]["legjobb_reteg"]])
    X_sel = eredmenyek[nyertes]["X"]
    print(f"reprezentáció: {nyertes}, {eredmenyek[nyertes]['legjobb_reteg']}. réteg")
else:
    X_sel = X_best
    print(f"reprezentáció: WavLM-base-plus, {best_layer}. réteg (4. blokk)")

r8 = su.run_experiment(X_sel, y,  tr_spk, te_spk, name="8 osztály", verbose=False)
r7 = su.run_experiment(X_sel, y7, tr_spk, te_spk, name="7 osztály", verbose=False)

su.compare_results([r8, r7], title="Semleges és nyugodt összevonva")
plt.show()

print(f"8 osztály: acc={r8['accuracy']:.3f}  makro-F1={r8['macro_f1']:.3f}")
print(f"7 osztály: acc={r7['accuracy']:.3f}  makro-F1={r7['macro_f1']:.3f}")
print(f"\nvéletlen találat: {1/8:.3f} → {1/7:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
su.plot_confusion(r8, ax=axes[0], title="8 osztály")
su.plot_confusion(r7, labels=["semleges+nyugodt"] + su.EMOTION_ORDER[2:],
                  ax=axes[1], title="7 osztály")
plt.tight_layout(); plt.show()

In [ ]:
# Mennyi a javulás azokon az osztályokon, amiket NEM vontunk össze?
from sklearn.metrics import f1_score

valtozatlan = [c for c in su.EMOTION_ORDER if c not in ("semleges", "nyugodt")]
f8 = f1_score(r8["y_true"], r8["y_pred"], labels=valtozatlan, average=None,
              zero_division=0)
f7 = f1_score(r7["y_true"], r7["y_pred"], labels=valtozatlan, average=None,
              zero_division=0)
tabla = pd.DataFrame({"8 osztály": f8, "7 osztály": f7}, index=valtozatlan).round(3)
tabla["változás"] = (tabla["7 osztály"] - tabla["8 osztály"]).round(3)
tabla.sort_values("változás", ascending=False)

> **Kérdések:**
>
> 1. Mennyit javult a makro-F1? Mennyi ebből „ingyen" jött attól, hogy kevesebb
>    osztály van (a véletlen találat 12,5%-ról 14,3%-ra nőtt)?
> 2. A **nem összevont** osztályok F1-értéke javult-e? Ha igen, miért? *(Ha a
>    modellnek nem kell két egymáshoz nagyon hasonló osztály közt döntenie,
>    a döntési határok a többi osztályra is tisztábbak lehetnek.)*
> 3. Tisztességes-e ezt a két számot egymás mellé tenni egy cikkben? Mit kellene
>    mindenképpen odaírni?
>
> **A valódi tanulság nem az, hogy „összevonással jobb lesz a szám".** Hanem az,
> hogy a **kategórialista tervezési döntés**. Ha az üzleti feladat szempontjából
> a semleges és a nyugodt ugyanaz, akkor eleve így kellett volna címkézni —
> és akkor az annotátorok egyetértése is magasabb lett volna.
>
> Ez ugyanaz a gondolat, mint az előadáson a feladatszűkítésről: egy jól
> megfogalmazott szűk feladat többet ér, mint egy rosszul definiált nyolcosztályos.

---
## Zárás — mi hiányzik még egy éles rendszerből

Amit ma megépítettünk, az egy **működő demó**. Egy éles rendszerhez ezek
hiányoznak:

| hiányzik | miért számít |
|---|---|
| a célhoz illő adat | spontán, a valós csatornán rögzített, a célnyelven |
| feladatszűkítés | 8 kategória helyett az, ami üzletileg tényleg kell |
| rétegsúlyozás / finomhangolás | fagyasztott egyetlen réteg helyett |
| kalibráció és bizonytalanság | mikor mondja a rendszer, hogy „nem tudom” |
| keresztkorpusz teszt | általánosít-e egyáltalán |
| jogi keret | az EU AI Act szerint hol szabad egyáltalán használni |

### Ha tovább akarsz menni

A 7-8-9. blokkban már kipróbáltuk a nagyobb modellt, a leave-one-speaker-out
kiértékelést és az osztályösszevonást. Innen tovább:

1. Vond be a szöveget is: futtass egy ASR-t (Whisper), és fűzd a szöveg-embeddinget
   az akusztikaihoz. Ez a challenge-rendszerek 76%-ának receptje.
2. Tanítsd újra kizárólag a **férfi** színészeken, és teszteld **női** beszélőkön —
   mit mutat ez a torzításról? (A 8. blokk nem szerinti bontása csak a tünetet
   mutatja, ez a kísérlet az okot.)
3. Egyetlen réteg helyett **tanulható súlyú rétegkombináció**: a 7. blokk
   rétegprofiljai megmutatják, hogy több réteg is hordoz információt.
4. Próbáld ki ugyanezt a pipeline-t egy **másik korpuszon** (EmoDB, CREMA-D),
   és teszteld keresztkorpusz módon: RAVDESS-en tanítva, máson tesztelve.
5. Cseréld a fagyasztott jellemzőket **LoRA-finomhangolásra** — mennyit hoz,
   és mennyivel tart tovább?